<a href="https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/12_observabilidad_evaluacion.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

🎬 [La escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap12) · 🧭 [Teoría del capítulo](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap12-observabilidad-evaluacion.html) · 🗺️ [El mapa](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html) · ⬅️ [Cap. 11](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/11_code_interpreter.ipynb) · ➡️ [Cap. 13](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/13_ab_testing.ipynb)

# Capítulo 12 — ¿Lo hace bien?

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** Dirección pregunta: *«¿Qué tal funciona Nova Assist?»*. Solo tenemos impresiones y algún pantallazo.

| | |
|---|---|
| **🧠 Componente agéntico** | Observabilidad y evaluación |
| **La necesidad** | Ver cada decisión del agente y medir su calidad (y su coste) de forma sistemática |
| **Lo que construimos** | Primero un juez con rúbrica en local; después, para producción: **Observability** (trazas OTel), **Evaluations** online con muestreo, histórico en Iceberg y **límites de coste**: presupuesto por conversación y *rate limit* en el Gateway |
| **🏠 En local** | ✅ Funciona a pequeña escala: un juez con rúbrica sobre unos pocos casos (sección 0). No escala a tráfico real ni sigue servicios distribuidos. |
| **☁️ Por qué AWS al salir a Internet** | Con `print()` y logs sueltos no se reconstruye una conversación con diez herramientas. Las trazas OTel siguen cada paso y las evaluaciones online con muestreo convierten la calidad en una métrica continua. |
| **En el ordenador sería…** | El **visor de eventos y los exámenes**: ver qué ha pasado y medir el rendimiento con pruebas repetibles |
| **Requisitos del caso** | RNF-05 |
| **Prerrequisito · duración** | Capítulo 11 · 60 min |
| **⚠️ Dónde se rompe la analogía** | El visor de eventos muestra lo que pasó; las evaluaciones con un LLM como juez **opinan** sobre lo que pasó. Hay que calibrar al juez. |

### ☁️ ¿Estás en Google Colab? Ejecuta primero esta celda

En tu equipo no hace nada. En Colab descarga el curso, monta tu **Google Drive** (ahí se guarda tu estado entre capítulos: `MyDrive/NovaAssist_curso`), instala las dependencias y la AgentCore CLI 0.31.1 y carga tus credenciales de AWS desde los **Secretos** de Colab. Tarda 2–3 minutos, porque cada notebook abre una máquina nueva: mientras tanto, lee la escena.

In [ ]:
# ☁️ Solo en Google Colab (en tu equipo no hace nada)
import os, sys, subprocess
if "google.colab" in sys.modules:
    CURSO = "/content/aws-bedrock-agentcore-course"
    if not os.path.isdir(CURSO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                        "https://github.com/farithjhg/aws-bedrock-agentcore-course.git", CURSO], check=True)
    os.chdir(f"{CURSO}/v7_aprender_haciendo/labs")
    sys.path.insert(0, os.getcwd())
    import colab_nova
    colab_nova.preparar(capitulo=12)

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(12)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(12)
# ponerse_al_dia(12)   # ← descomenta si llegas tarde o algo se rompió

## 0. El concepto en local: un juez con rúbrica

Evaluar un agente es comparar lo que hace con lo que debería hacer. En pequeño: tres casos de referencia, la respuesta del agente y un modelo que hace de **juez**.

> ✋ **Predice antes de ejecutar.** Tres casos de referencia: ¿cuántos aprobará el juez? Y otra: ¿qué **no** puede ver el juez de lo que pasó por dentro de cada conversación?

In [ ]:
import json, re
from strands import Agent
from strands.models import BedrockModel

ds = json.loads((DATOS_DIR / "tiendanova.json").read_text(encoding="utf-8"))
casos_locales = [c for c in ds["eval_casos"] if c["caso_id"] in ("E01", "E07", "E10")]
RUBRICA = ("Eres un evaluador. Compara la respuesta del asistente con la respuesta esperada. Responde SOLO con JSON: "
           '{"correcta": true o false, "motivo": "una frase"}')
TOK = token_cliente("C-001")
for c in casos_locales:
    invocar(c["prompt"], token=TOK)
    juez = Agent(model=BedrockModel(model_id=state_get("MODEL_ID"), region_name=region(), temperature=0),
                 system_prompt=RUBRICA, callback_handler=None)
    veredicto = str(juez(f"Pregunta: {c['prompt']}\nRespuesta esperada: {c['respuesta_esperada']}\n"
                         f"Respuesta del asistente: {ultima_respuesta()}"))
    m = re.search(r"\{.*\}", veredicto, re.S)
    print(f"⚖️ {c['caso_id']}:", json.loads(m.group(0)) if m else veredicto, "\n")

Funciona con tres casos y alguien mirando. En producción la situación es otra:
- hay **miles de conversaciones al día**;
- cada una atraviesa **varios servicios** (Runtime, Memory, Gateway, Lambdas, S3 Vectors);
- **nadie está mirando**.

Hace falta reconstruir cada decisión de principio a fin (**trazas distribuidas**), evaluar **tráfico real con muestreo** y guardar el **histórico**. Eso es lo que construye el resto del capítulo.

El juez ha opinado sobre tres respuestas. De lo que pasó por dentro (qué herramienta, qué decidió Policy, cuánto tardó Athena) no sabe nada.

> 🧭 **¿Por qué ha pasado?** Abre la teoría: [**El visor de eventos y los exámenes**](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap12-observabilidad-evaluacion.html). Compara qué se reconstruye con logs sueltos y qué con una traza completa, pasa el examen y decide qué parte del tráfico evaluar. Vuelve cuando sepas responder a esta pregunta: **¿qué necesitas para saber qué pasó en una conversación que atravesó cinco servicios y nadie miró, y por qué no se evalúa el 100 % del tráfico?**

**En dos líneas:** **observabilidad** es poder reconstruir cada decisión del agente (trazas del modelo, las herramientas y la memoria); **evaluación** es medir la calidad con casos de referencia y jueces (código o un LLM con rúbrica), *offline* antes de desplegar y *online* sobre tráfico real con muestreo. Es el bucle de retroalimentación del agente: sin medición no hay mejora, y como el juez es un modelo, hay que calibrarlo.

## 1. Observabilidad

Requisito: **CloudWatch Transaction Search** activado (capítulo 0). Abre después CloudWatch → *GenAI Observability* y recorre una sesión completa: modelo, herramientas del Gateway, Memory y Code Interpreter.

In [ ]:
cli('agentcore traces list')

In [ ]:
cli('agentcore logs --since 30m --level error')

## 2. Conjunto de prueba desde Iceberg

El conjunto de evaluación vive en la tabla Iceberg `eval_casos` (capítulo 0, Parte C): casos de referencia con herramientas y documento esperados, y casos adversariales. Ejecutamos los de los clientes con usuario en Cognito (C-001 y C-002).

In [ ]:
casos = [c for c in leer_tabla("eval_casos") if c["cliente"] in ("C-001", "C-002")]
tokens = {u: token_cliente(u) for u in ("C-001", "C-002")}
ejecuciones = []
for c in casos:
    print(f"— {c['caso_id']} [{c['categoria']}] esperado: {c['respuesta_esperada']}")
    sid = invocar(c["prompt"], token=tokens[c["cliente"]])
    ejecuciones.append({"caso_id": c["caso_id"], "session_id": sid})
state_set(EVAL_SESSIONS=[e["session_id"] for e in ejecuciones])

## 3. Evaluación bajo demanda

In [ ]:
for sid in state_get("EVAL_SESSIONS")[:3]:
    cli_args("run", "eval", "--runtime", "NovaAgent", "--session-id", sid,
             "--evaluator", "Builtin.GoalSuccessRate", "Builtin.Helpfulness")

Añade `Builtin.ToolSelectionAccuracy`, que trabaja a nivel de llamada a herramienta, para medir si el agente eligió la herramienta correcta. Si la evaluación no encuentra la sesión, espera unos segundos (las trazas tardan en estar disponibles) y repite.

## 4. Evaluación online continua

In [ ]:
# Los nombres de configuración usan guiones bajos. Configura el porcentaje de muestreo (--help).
cli('agentcore add online-eval --help')

In [ ]:
cli_args("add", "online-eval", "--name", "nova_online_eval", "--runtime", "NovaAgent",
         "--evaluator", "Builtin.GoalSuccessRate", "Builtin.Helpfulness",
         "--sampling-rate", "20", "--enable-on-create")
assert cli("agentcore validate") == 0
cli("agentcore deploy --yes")

Para pausarla y reanudarla: `agentcore pause online-eval nova_online_eval` y `agentcore resume online-eval nova_online_eval`.

> Evaluamos el **20 %** de las sesiones: evaluar el 100 % del tráfico con jueces LLM puede duplicar el consumo de inferencia (doc §8). **Muestrea.**

## 5. Evaluación del retrieval con histórico en Iceberg

El *recall@k* del capítulo 8 se convierte en una métrica con histórico: cada ejecución se añade a la tabla Iceberg `eval_retrieval`. Así puedes comparar modelos de *embeddings*, troceados o versiones de documentos a lo largo del tiempo.

In [ ]:
from datetime import datetime, timezone
import rag_nova

casos_rag = [c for c in leer_tabla("eval_casos") if c["documento_esperado"]]
ahora = datetime.now(timezone.utc).replace(tzinfo=None)
filas = []
for c in casos_rag:
    docs = [r["doc_id"] for r in rag_nova.buscar(c["prompt"], top_k=5)]
    filas.append({"ejecucion": ahora, "caso_id": c["caso_id"], "modelo_embedding": rag_nova.MODELO_EMBEDDING,
                  "documento_esperado": c["documento_esperado"], "top5": ",".join(docs),
                  "acierto_top1": docs[:1] == [c["documento_esperado"]],
                  "acierto_top3": c["documento_esperado"] in docs[:3]})
escribir_tabla("eval_retrieval", filas, modo="append")
for fila in athena("SELECT ejecucion, modelo_embedding, avg(CAST(acierto_top3 AS double)) AS recall_at_3 "
                   "FROM eval_retrieval GROUP BY ejecucion, modelo_embedding ORDER BY ejecucion"):
    print(fila)

## 6. Evaluador propio (opcional)

Crea un evaluador LLM "cumplimiento de la política de devoluciones" con criterios explícitos: cita el documento, no promete plazos fuera de la política y escala cuando procede. Añade *ground truth* con la secuencia de herramientas esperada para cada caso del conjunto de prueba.

## 6b. Límites de coste: lo que el código decide, no el modelo

Medir el coste (taxímetro, trazas) no lo **limita**. Dos límites deterministas:

1. **Presupuesto por conversación** (`presupuesto.py`): el agente lleva la cuenta de los *tokens* de cada conversación en su estado, que Memory guarda con ella. Si se supera el presupuesto, responde sin llamar al modelo y deriva a una persona. Es la respuesta al «Black Friday» del capítulo 16.
2. **Rate limit en el Gateway**: cada aplicación que llama a las herramientas (identificada por el `client_id` de su token) tiene un máximo de llamadas por minuto. Un agente en bucle no deja sin servicio a la app de repartidores (capítulo 7).

In [ ]:
%%writefile NovaAssist/app/NovaAgent/presupuesto.py
"""Presupuesto de tokens por conversación (capítulo 12): un límite DETERMINISTA al coste de cada conversación."""
import os

PRESUPUESTO_TOKENS = int(os.environ.get("NOVA_PRESUPUESTO_TOKENS", "150000"))
MENSAJE = ("Esta conversación ya es muy larga. Para atenderte mejor, te paso con una persona del equipo: "
           "te responderá en menos de 24 horas laborables.")


def consumidos(agent) -> int:
    return int(agent.state.get("tokens_conversacion") or 0)


def agotado(agent) -> bool:
    return consumidos(agent) >= PRESUPUESTO_TOKENS


def anotar(agent, session_manager, result) -> int:
    """Suma los tokens de esta invocación al estado del agente, que Memory guarda con la conversación."""
    uso = dict(result.metrics.accumulated_usage)
    total = consumidos(agent) + sum(int(uso.get(k, 0)) for k in
                                    ("inputTokens", "outputTokens", "cacheReadInputTokens", "cacheWriteInputTokens"))
    agent.state.set("tokens_conversacion", total)
    session_manager.sync_agent(agent)
    return total

In [ ]:
%%writefile NovaAssist/app/NovaAgent/main.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.agent.conversation_manager import SlidingWindowConversationManager
from strands_tools.code_interpreter import AgentCoreCodeInterpreter

import presupuesto
import prompts
from contexto import VENTANA_MENSAJES, system_prompt
from gateway import cliente_gateway, token_gateway
from identidad import actor_desde_contexto
from memoria import crear_session_manager, retrieval_config
from modelo import crear_modelo
from respuesta import respuesta
from settings import REGION

app = BedrockAgentCoreApp()
code_interpreter = AgentCoreCodeInterpreter(region=REGION)


@app.entrypoint
def invoke(payload, context):
    actor_id = actor_desde_contexto(context)   # identidad verificada; payload["cliente_id"] se IGNORA
    if not actor_id:
        return {"respuesta": "No se ha podido identificar al cliente.", "herramientas": [], "tokens": {}}
    version = getattr(prompts, "VERSION_CODIGO", None)
    gateway = cliente_gateway(token_gateway())
    with gateway, crear_session_manager(actor_id, context.session_id, retrieval_config()) as sm:
        agent = Agent(
            model=crear_modelo(),
            system_prompt=system_prompt(payload),
            tools=gateway.list_tools_sync() + [code_interpreter.code_interpreter],
            session_manager=sm,
            conversation_manager=SlidingWindowConversationManager(window_size=VENTANA_MENSAJES),
            callback_handler=None,
        )
        if presupuesto.agotado(agent):   # límite de coste por conversación: lo decide el código, no el modelo
            return {"respuesta": presupuesto.MENSAJE, "herramientas": ["presupuesto_agotado"], "tokens": {},
                    "actor": actor_id, "version_codigo": version}
        result = agent(payload.get("prompt", ""))
        presupuesto.anotar(agent, sm, result)
    return respuesta(result, actor=actor_id, version_codigo=version)


if __name__ == "__main__":
    app.run()

> ✋ **Predice antes de ejecutar.** Con un presupuesto de solo 3.000 *tokens* por conversación, ¿cuántos mensajes podrá enviar Laura antes de que el agente la derive?

In [ ]:
# Prueba local con un presupuesto ridículo para ver el límite en acción
os.environ["NOVA_PRESUPUESTO_TOKENS"] = "3000"
os.environ["NOVA_GATEWAY_TOKEN"] = token_m2m()
cabeceras = {"Authorization": f"Bearer {token_cliente('C-001')}"}
sid = new_session_id()
probar_local("¿Dónde está mi pedido TN-1001?", session_id=sid, headers=cabeceras)
probar_local("¿Y quién lo lleva?", session_id=sid, headers=cabeceras)
PRESUPUESTO_OK = "presupuesto_agotado" in ultimas_herramientas()
print("¿Ha cortado la conversación al superar el presupuesto?", PRESUPUESTO_OK)
del os.environ["NOVA_PRESUPUESTO_TOKENS"], os.environ["NOVA_GATEWAY_TOKEN"]

En el Runtime, el presupuesto por defecto es de 150.000 *tokens* por conversación (ajústalo con la variable `NOVA_PRESUPUESTO_TOKENS`). Desplegamos:

In [ ]:
cli("agentcore deploy --yes")

Ahora el *rate limit*. Para verlo actuar lo creamos con un límite muy bajo (5 llamadas por minuto por aplicación), lo provocamos y lo subimos a un valor de producción en la misma celda, para no dejar al agente sin herramientas.

> ✋ **Predice antes de ejecutar.** Ocho llamadas seguidas con un límite de cinco por minuto: ¿cuántas fallarán? Los cambios de límite tardan hasta 30 segundos en aplicarse y, si el servicio de límites falla, el Gateway **deja pasar** (*fail-open*).

In [ ]:
import time
import boto3
ctl = boto3.client("bedrock-agentcore-control", region_name=region())
gw = state_get("GATEWAY_ARN").rsplit("/", 1)[-1]
CLAVE = "$.context.jwt.client_id"           # una cuota por aplicación que llama (client_id del token)

def entradas(por_minuto):
    return [{"dimensions": {CLAVE: "*"}, "requests": [{"rate": por_minuto, "period": "minute"}]}]

existente = next((r for r in ctl.list_gateway_rate_limits(gatewayIdentifier=gw).get("rateLimits", [])
                  if r["dimensionKeys"] == [CLAVE]), None)
if existente:
    rl = ctl.update_gateway_rate_limit(gatewayIdentifier=gw, rateLimitId=existente["rateLimitId"], entries=entradas(5))
else:
    rl = ctl.create_gateway_rate_limit(gatewayIdentifier=gw, dimensionKeys=[CLAVE], entries=entradas(5),
                                       description="Nova: llamadas por minuto por aplicación")
state_set(RATE_LIMIT_ID=rl["rateLimitId"])
try:
    print("Esperando 35 s a que se aplique el límite…"); time.sleep(35)
    tok = token_m2m()
    codigos = [mcp_call("PedidosTarget___consultar_envio", {"pedido_id": "TN-1001"}, token=tok)["http"] for _ in range(8)]
    print("Códigos HTTP:", codigos)
    RATE_LIMIT_OK = 429 in codigos
finally:
    ctl.update_gateway_rate_limit(gatewayIdentifier=gw, rateLimitId=rl["rateLimitId"], entries=entradas(120))
    print("Límite restaurado a 120 llamadas por minuto por aplicación")

Si aparecen **429**, el Gateway ha frenado a la aplicación cuando superó su cuota. Si no aparece ninguno, el cambio aún no se había propagado o el servicio de límites ha dejado pasar (*fail-open*): repite la celda dentro de un minuto. El límite es una **protección**, no una garantía al milisegundo.

## 7. Lo que revelan los datos

Además de evaluar al agente, la plataforma de datos permite mirar el **negocio**. Tasa de devolución por producto en los últimos 90 días:

> ✋ **Predice antes de ejecutar.** ¿Qué producto de TiendaNova crees que tiene la mayor tasa de devolución? Apúntalo antes de lanzar la consulta.

In [ ]:
for fila in athena("""
    WITH entregas AS (
      SELECT l.producto_id, l.nombre, count(DISTINCT p.pedido_id) AS pedidos
      FROM pedidos p JOIN lineas_pedido l ON p.pedido_id = l.pedido_id
      WHERE p.estado = 'ENTREGADO' AND p.fecha_entrega >= current_date - interval '90' day
      GROUP BY 1, 2),
    dev AS (
      SELECT producto_id, count(*) AS devoluciones FROM devoluciones
      WHERE fecha >= current_date - interval '90' day GROUP BY 1)
    SELECT e.producto_id, e.nombre, e.pedidos, coalesce(d.devoluciones, 0) AS devoluciones,
           round(100.0 * coalesce(d.devoluciones, 0) / e.pedidos, 1) AS tasa_pct
    FROM entregas e LEFT JOIN dev d ON e.producto_id = d.producto_id
    ORDER BY tasa_pct DESC LIMIT 5"""):
    print(fila)

Lo esperable es que un producto destaque muy por encima del resto (el dataset lo incluye a propósito). Guarda la pregunta: volverá en el capítulo 14.

In [ ]:
taximetro_historico()

## Para pensar

- ¿Qué umbral de `GoalSuccessRate` exigirías para pasar a producción?
- ¿Qué alarmas de CloudWatch crearías (errores, latencia, `ActiveSessionCount`, coste)?

## 🔎 Lo que acabamos de descubrir

Dos hallazgos. Los evaluadores señalan respuestas mejorables, pero **cambiar el prompt para todos los clientes da miedo**. Y los datos muestran un **pico de devoluciones del Robot aspirador S9**.

In [ ]:
nota = "presupuesto de tokens por conversación en el código" + ("; rate limit por aplicación en el Gateway" if RATE_LIMIT_OK else "")
deuda("D6", "pagada" if PRESUPUESTO_OK else "parcial", 12,
      nota if PRESUPUESTO_OK else "el presupuesto por conversación no cortó en la prueba: revisa presupuesto.py")
cuaderno()
taximetro_cerrar(12)
guardar_partida(12)

📚 **Para repasar más adelante:** la teoría de este capítulo está en [`teoria/cap12-observabilidad-evaluacion.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap12-observabilidad-evaluacion.html) y el mapa completo en [`teoria/index.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html). La referencia por componente es la Parte I del documento.

---
➡️ **Siguiente capítulo:** `13_ab_testing.ipynb` — *Mejorar sin romper*

---
**Siguiente:** ➡️ [Capítulo 13 · Mejorar sin romper](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/13_ab_testing.ipynb) · 🎬 [su escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap13). En Colab, tu estado ya está en Drive: el capítulo siguiente lo recupera solo.